🚀 Great. You've completed:

Day 1 → EDA + popularity baseline
Day 2 → Weighted ratings
Day 3 → User collaborative filtering
Day 4 → Item collaborative filtering
Day 5 → SVD / Matrix factorization
Day 6 → Top-N recommendation engine
Day 7 → Content-based recommendation
Day 8 → Hybrid recommendation system
Day 9 → Production-style modular project structure

Now we solve a major production problem.

🚀 Day 10 — Model Persistence (pickle / joblib)

Right now:

Every time you run:

main.py

your system:

Loads data
↓
Retrains SVD
↓
Rebuilds similarity matrix
↓
Then recommends

Problem:

Training repeatedly
=
Slow
Wasteful
Not production-ready

Real systems:

Train once
↓
Save model
↓
Load instantly later

This is called:

🧠 Model Persistence


---

Goal Today

Learn to:

✅ Save trained models
✅ Load models later
✅ Avoid retraining
✅ Make API-ready architecture


---

🧠 Two common approaches

1. pickle

Python built-in serialization.

2. joblib

Optimized for large ML objects.

We'll use:

joblib

because ML models can become large.


---

Step 1 — Install joblib

pip install joblib


---

Step 2 — Create models/ folder

Project structure:

recommendation_system/

models/

This folder stores:

trained_model.pkl
similarity.pkl


---

Step 3 — Save SVD model

Update collaborative.py

import joblib

from surprise import SVD
from surprise import Dataset
from surprise import Reader


def train_svd_model(ratings):

    reader = Reader(
        rating_scale=(1,5)
    )

    data = Dataset.load_from_df(
        ratings[
            [
                'user_id',
                'movie_id',
                'rating'
            ]
        ],
        reader
    )

    trainset = data.build_full_trainset()

    model = SVD()

    model.fit(trainset)

    # SAVE MODEL

    joblib.dump(
        model,
        'models/svd_model.pkl'
    )

    print(
        "SVD model saved"
    )

    return model


---

Step 4 — Load model later

Add:

def load_svd_model():

    model = joblib.load(
        'models/svd_model.pkl'
    )

    print(
        "SVD model loaded"
    )

    return model

Now:

Train once
↓
Load many times


---

Step 5 — Save similarity matrix

Update content_based.py

import joblib

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics.pairwise import cosine_similarity


def build_content_similarity(movies):

    genre_columns = movies.columns[5:]

    movies['genres'] = movies[
        genre_columns
    ].apply(
        lambda x:' '.join(
            x.index[x==1]
        ),
        axis=1
    )

    cv = CountVectorizer()

    movie_vectors = cv.fit_transform(
        movies['genres']
    )

    similarity = cosine_similarity(
        movie_vectors
    )

    # SAVE MATRIX

    joblib.dump(
        similarity,
        'models/similarity.pkl'
    )

    print(
        "Similarity matrix saved"
    )

    return similarity


---

Step 6 — Load similarity later

def load_similarity():

    similarity = joblib.load(
        'models/similarity.pkl'
    )

    print(
        "Similarity matrix loaded"
    )

    return similarity


---

Step 7 — Update main.py

First run:

from src.collaborative import (
    train_svd_model
)

from src.content_based import (
    build_content_similarity
)

model = train_svd_model(
    ratings
)

similarity = build_content_similarity(
    movies
)

After models exist:

from src.collaborative import (
    load_svd_model
)

from src.content_based import (
    load_similarity
)

model = load_svd_model()

similarity = load_similarity()


---

🧠 What changed now?

Before:

Run app
↓
Retrain everything

Now:

Run app
↓
Load trained models instantly

Huge production improvement.


---

Why this matters in real systems

Imagine:

Netflix retrains model
every API request

Impossible.

Instead:

Offline training
↓
Save model
↓
Deploy trained artifacts
↓
Serve predictions quickly


---

🧠 Important production concepts learned

1. Serialization

Convert Python object into storable format.


---

2. Model artifacts

Saved ML objects:

.pkl
.joblib


---

3. Offline training

Train separately from inference.


---

4. Inference pipeline

Load model
↓
Predict quickly

instead of training.


---

Small improvement (recommended)

Instead of manually switching:

train
vs
load

use:

import os

Example:

if os.path.exists(
    'models/svd_model.pkl'
):

    model = load_svd_model()

else:

    model = train_svd_model(
        ratings
    )

Now system automatically:

Train once
↓
Load afterwards


---

🎯 Homework

1. Save both:

svd_model.pkl
similarity.pkl

Check:

Did files appear inside models/ ?


---

2. Delete notebook kernel and rerun

Check:

Does it load instantly now?


---

3. Think about this

Right now:

System works only in terminal

Question:

How can users request recommendations through URLs?

That becomes:

🚀 Day 11 — FastAPI Recommendation API

